# 01 · Entorno Colab, accesos y persistencia

**Alcance: fase 1, etapa 1 exclusivamente.** No descarga datos ni pesos, no prepara splits, no instancia modelos y no ejecuta inferencia o entrenamiento.

Estado al entregar (2026-10-28): validación local; **Colab aún no ejecutado**. Consulte `docs/project/validacion_etapa1.md` y `docs/project/accesos_y_licencias.md`. Sen1Floods11 y DINOv3 tienen bloqueos documentados; la prueba de persistencia es independiente.

## Preparación de una sesión limpia
1. Use un runtime **Python 3.12 / Linux x86_64** de Colab. Seleccione GPU si está disponible; se registra la realmente asignada, sin prometer suficiencia para etapas futuras.
2. En el repositorio local ejecute `python scripts/export_colab_bundle.py`. Suba `artifacts/auto-prompt-sam2-stage1.zip` a su Drive. Conserve el SHA-256 mostrado por el exportador.
3. Copie también este notebook a Colab. Configure las rutas y el hash del ZIP abajo. La instalación sucede en `/content/auto-prompt-venv`, aislada del kernel de Colab. Todas las validaciones usan ese intérprete en procesos nuevos.
4. Para comprobar acceso autorizado a DINOv3, el titular debe aceptar su licencia, obtener aprobación y crear el secreto **HF_TOKEN** en Colab con permiso de lectura de ese repositorio. Autorice este notebook a leerlo. Nunca escriba el token en una celda, URL, salida o archivo. Si no existe, se comprueba solo acceso anónimo y se registra el bloqueo.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage1.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Copiar el hash no secreto producido por export_colab_bundle.py.
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
ACTION = 'prepare'  # Tras reiniciar completamente Colab, cambiar a 'recover'.
RUN_ID = ''  # Vacío: busca ticket persistente compatible; opcional: ID mostrado al preparar.
REPO = Path('/content/auto-prompt-sam2-stage1')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
assert EXPECTED_BUNDLE_SHA256, 'Configure el hash del ZIP exportado.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256, 'ZIP incompatible/corrupto'
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        target = (REPO / item.filename).resolve()
        assert target.is_relative_to(REPO.resolve()), 'Ruta insegura en el ZIP'
    archive.extractall(REPO)
print('Fuente verificada. No se han adquirido datos ni pesos.')


## Bootstrap idempotente
El lock incluye versiones y hashes de dependencias transitivas, resueltos para Python 3.12/Linux. PyTorch 2.7.1 usa CUDA 12.6 en el wheel Linux de PyPI y torchvision 0.22.1 es su pareja oficial. SAM se instala desde un commit exacto, sin su extensión CUDA opcional: cualquier posprocesamiento que la requiera queda pendiente de evaluación en su etapa. El código de SAM se instala; **sus pesos no se descargan**.

El script detiene versiones/plataformas incompatibles. No actualice automáticamente el lock ante un error. La instalación real Linux/Colab sigue pendiente hasta ejecutar esta celda correctamente.


In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts/bootstrap_colab.py'),
                '--venv', str(PYTHON.parent.parent)], check=True)
config = json.loads((REPO / 'configs/environment.yaml').read_text())
child_env = dict(os.environ, AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT,
                 PYTHONHASHSEED=str(config['seed']), CUBLAS_WORKSPACE_CONFIG=':4096:8',
                 HF_HUB_DISABLE_TELEMETRY='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1')
subprocess.run([str(PYTHON), '-m', 'pip', 'check'], check=True, env=child_env)
subprocess.run([str(PYTHON), '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'],
               check=True, cwd=REPO, env=child_env)


## Accesos: solo metadatos y solicitudes HEAD
Se verifica cada revisión fijada, el nombre del archivo y la autorización HEAD sin seguir la redirección al almacenamiento de pesos. GET se limita a API de metadatos y dos catálogos STAC pequeños. Las respuestas guardadas usan una lista explícita de campos; no se guardan cabeceras, URLs firmadas, cuentas ni excepciones con credenciales.

Un HTTP 401/403 no determina si el titular ya tiene permiso: puede faltar el secreto o el alcance del token. La presencia pública de la ficha tampoco prueba autorización. La licencia `proprietary` del catálogo Sen1Floods11 requiere aclaración del publicador para S2Hand y LabelHand; no basta cambiar una bandera de configuración.


In [ ]:
from google.colab import userdata
_secret = None
try:
    _secret = userdata.get('HF_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print('HF_TOKEN no disponible: se verifica acceso anónimo.')
try:
    # input via stdin: el token no aparece en argv, entorno, historial ni archivos.
    subprocess.run([str(PYTHON), str(REPO / 'scripts/check_access.py'),
                    '--output', str(Path(PERSIST_ROOT) / 'environment'), '--token-stdin'],
                   input=_secret or '', text=True, check=True, env=child_env)
finally:
    _secret = None
    del _secret


## Preparar o recuperar la prueba pequeña
`prepare` crea un run nuevo, fija semilla 42, comprueba importaciones sin construir modelos y conserva inventario (CPU, RAM, disco, GPU/VRAM, driver, CUDA, paquetes), fuente exacta y artefacto de prueba. Deja deliberadamente un archivo incompleto sin marcador. El run queda `running` simulando una desconexión.

**Después de `prepare`: anote el run_id, desconecte y elimine el runtime de Colab.** Abra un runtime nuevo, vuelva a montar el mismo Drive y ejecute desde el inicio con `ACTION = 'recover'` y el mismo ZIP, hash y raíz. No basta ejecutar `recover` en la misma sesión para acreditar el reinicio de Colab. El ticket persistente evita depender de variables del kernel anterior.

`recover` exige identidad/configuración/revisiones/fuente y paquetes compatibles; verifica el SHA-256, rechaza la escritura incompleta y una identidad distinta. Registra nuevamente hardware y la transición `running → interrupted → running → complete`. **`complete` identifica solo el ensayo de persistencia**, no el cierre de la etapa ni permisos de uso. Nunca ejecute dos escritores sobre el mismo run.


In [ ]:
assert ACTION in ('prepare', 'recover')
command = [str(PYTHON), str(REPO / 'scripts/validate_environment.py'), ACTION, '--colab']
if ACTION == 'recover' and RUN_ID:
    command += ['--run-id', RUN_ID]
subprocess.run(command, check=True, env=child_env, cwd=REPO)


## Evidencia y cierre manual de la etapa
Los resultados completos están en `<PERSIST_ROOT>/environment/runs/<run_id>/`: `environment`, `smoke`, `source_snapshot`, `recovery_validation` y `state`. Cada artefacto tiene generaciones inmutables con `payload` y `COMPLETE.json`; la lectura verifica tamaño, hash e identidad. Un marcador roto nunca hace válido un archivo. Si la última generación es inválida se recupera la última completa compatible, sin borrar la fallida. Un volumen remoto necesita lectura posterior al remontaje; el ensayo local no prueba durabilidad de Drive.

Presupuesto configurable: 5 GiB datos, 4 pesos, 12 cachés, 5 checkpoints, 2 predicciones y 2 de reserva (30 GiB total), más 10 GiB locales para instalar. Son reservas de planificación, **no consumos medidos**. `disk_usage` en Drive puede reflejar el montaje y no la cuota real: compruebe también la cuota en Drive antes de continuar. No se escribe relleno ni se adquieren los contenidos reservados.

Para concluir: conservar evidencia de instalación limpia y recuperación en runtime nuevo, GPU/memoria realmente disponibles, aprobación HEAD de DINOv3 con el secreto y aclaración de licencia de Sen1Floods11 con fuente verificable. Actualizar `docs/project/estado_actual_fase1.md` y el cuaderno. Sin esos puntos la etapa sigue **Bloqueada**, aunque pase el ensayo. Este notebook termina aquí: no inicia la etapa 2.
